In [1]:
import os, glob
import numpy as np
from keras.preprocessing.image import load_img, img_to_array
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from sklearn.model_selection import train_test_split

Using TensorFlow backend.


In [2]:
img_path = 'input/train'
img_ext = '.jpg'
img_form = img_path + '/{}' + img_ext

lbl_path = 'input/train/segmentation'
lbl_ext = '.png'
lbl_form = lbl_path + '/{}' + lbl_ext

In [3]:
img_names = glob.glob(img_path + '/*' + img_ext)
img_names = [names.replace('\\', '/') for names in img_names]
img_names = [names.split('/')[-1].split('.')[0] for names in img_names]
print('Found {} images.'.format(len(img_names)))

Found 17 images.


In [4]:
from u_net import get_unet_128
input_size, model = get_unet_128()

W0826 21:51:24.512046 19448 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0826 21:51:24.528044 19448 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0826 21:51:24.532044 19448 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0826 21:51:24.561044 19448 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:174: The name tf.get_default_session is deprecated. Please use tf.compat.v1.get_default_session instead.

W0826 21:51:24.563043 19

In [5]:
train, valid = train_test_split(img_names, test_size=0.10, random_state=42)

In [6]:
def train_generator():
    while True:
        for start in range(0, len(train)):
            x_batch = []
            y_batch = []
    
            id = train[start]
            img_path = img_form.format(id)
            mask_path = lbl_form.format(id)
            img = load_img(img_path,color_mode='rgb', target_size=(input_size,input_size))
            mask = load_img(mask_path, color_mode='grayscale', target_size=(input_size,input_size))
            img = img_to_array(img)
            mask = img_to_array(mask)
            x_batch.append(img)
            y_batch.append(mask)
            #print('x:{}, y:{}'.format(img.shape, mask.shape))
            
            x_batch = np.array(x_batch, np.float32) / 255
            y_batch = np.array(y_batch, np.float32) / 255
            yield x_batch, y_batch

In [7]:
def valid_generator():
    while True:
        for start in range(0, len(valid)):
            x_batch = []
            y_batch = []

            id = valid[start]    
            img_path = img_form.format(id)
            mask_path = lbl_form.format(id)
            img = load_img(img_path,color_mode='rgb', target_size=(input_size,input_size))
            mask = load_img(mask_path, color_mode='grayscale', target_size=(input_size,input_size))
            img = img_to_array(img)
            mask = img_to_array(mask)
            x_batch.append(img)
            y_batch.append(mask)
            x_batch = np.array(x_batch, np.float32) / 255
            y_batch = np.array(y_batch, np.float32) / 255
            yield x_batch, y_batch


In [8]:
callbacks = [
    EarlyStopping(
        monitor='val_dice_loss',
        patience=8,
        verbose=1,
        min_delta=1e-4,
        mode='max'),
    ReduceLROnPlateau(monitor='val_dice_loss',
                               factor=0.5,
                               patience=4,
                               verbose=1,
                               epsilon=1e-5,
                               mode='max'),
             ModelCheckpoint(monitor='val_dice_loss',
                             filepath='weights/best_weights.hdf5',
                             save_best_only=True,
                             save_weights_only=True,
                             mode='max'),
             TensorBoard(log_dir='logs')]

c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\callbacks.py:1065: UserWarning: `epsilon` argument is deprecated and will be removed, use `min_delta` instead.
  warnings.warn('`epsilon` argument is deprecated and '


In [9]:
model.fit_generator(generator=train_generator(),
                    steps_per_epoch=15,
                    epochs=20,
                    verbose=2,
                    callbacks=callbacks,
                    validation_data=valid_generator(),
                    validation_steps=2)

W0826 21:51:31.881819 19448 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\callbacks.py:850: The name tf.summary.merge_all is deprecated. Please use tf.compat.v1.summary.merge_all instead.

W0826 21:51:31.883818 19448 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\callbacks.py:853: The name tf.summary.FileWriter is deprecated. Please use tf.compat.v1.summary.FileWriter instead.



Epoch 1/20
 - 15s - loss: 0.7142 - dice_loss: 0.5686 - val_loss: 5.4945 - val_dice_loss: 0.4179
Epoch 2/20
 - 11s - loss: 0.4222 - dice_loss: 0.7260 - val_loss: 2.5235 - val_dice_loss: 0.5154
Epoch 3/20
 - 12s - loss: 0.3324 - dice_loss: 0.7804 - val_loss: 1.1852 - val_dice_loss: 0.2752
Epoch 4/20
 - 11s - loss: 0.3786 - dice_loss: 0.7673 - val_loss: 3.3790 - val_dice_loss: 0.5006
Epoch 5/20
 - 10s - loss: 0.2550 - dice_loss: 0.8320 - val_loss: 1.2771 - val_dice_loss: 0.7157
Epoch 6/20
 - 10s - loss: 0.2218 - dice_loss: 0.8562 - val_loss: 0.4652 - val_dice_loss: 0.7303
Epoch 7/20
 - 10s - loss: 0.2130 - dice_loss: 0.8649 - val_loss: 1.0512 - val_dice_loss: 0.3370
Epoch 8/20
 - 10s - loss: 0.1894 - dice_loss: 0.8827 - val_loss: 0.6037 - val_dice_loss: 0.6398
Epoch 9/20
 - 10s - loss: 0.1463 - dice_loss: 0.9073 - val_loss: 0.3407 - val_dice_loss: 0.7998
Epoch 10/20
 - 10s - loss: 0.1252 - dice_loss: 0.9220 - val_loss: 4.9849 - val_dice_loss: 0.4202
Epoch 11/20
 - 10s - loss: 0.1335 - dic

In [10]:
model.save('my_model.h5')